In [4]:
#!/usr/bin/env Rscript

## ------------------------------------------------------------
## STEP1-variable_Distance_MDS.R
## 変数間 距離 & MDS 計算（クラスタリング無し）
## - A/B/C/D の4データセットに対して
##   * 変数間相関行列
##   * raw距離 d_raw_var = 1 - r
##   * 欠損補正距離 d_corr_var
##   * metric MDS (cmdscale)
##   * non-metric MDS (isoMDS)
## を計算して保存する
## ------------------------------------------------------------

suppressPackageStartupMessages({
  library(MASS)  # isoMDS
})

## 再現性確保
set.seed(42)

## ===== 0. ルートパスの設定 =====
ROOT <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"

## 入力ディレクトリ
input_dir <- file.path(ROOT, "data", "for_MDS_STEP2")

## 出力ルート（変数間専用）
out_root_var <- file.path(ROOT, "data", "calc_core_var", "01_distance_mds_variables")

## 今日の日付（YYYYMMDD）
date_tag <- format(Sys.Date(), "%Y%m%d")

## MDS 次元数（必要に応じて変更可）
k_mds_default <- 10

## ===== 対象データセット一覧（A〜D） =====
datasets <- list(
  A = list(
    id        = "A",
    input_file = "preprocessed_features_OH.csv",
    subdir     = "A_OH_plus_others"
  ),
  B = list(
    id        = "B",
    input_file = "preprocessed_features_FP.csv",
    subdir     = "B_FP_plus_others"
  ),
  C = list(
    id        = "C",
    input_file = "preprocessed_features_OH_only.csv",
    subdir     = "C_OH_only"
  ),
  D = list(
    id        = "D",
    input_file = "preprocessed_features_FP_only.csv",
    subdir     = "D_FP_only"
  )
)

## ------------------------------------------------------------
## ユーティリティ: 数値列だけを抽出
## ------------------------------------------------------------
select_numeric_columns <- function(df) {
  is_num <- vapply(df, is.numeric, logical(1))
  if (!any(is_num)) {
    stop("[ERROR] No numeric columns found in the data.")
  }
  df_num <- df[, is_num, drop = FALSE]
  return(df_num)
}

## ------------------------------------------------------------
## メイン処理: 1データセット分（変数間）
## ------------------------------------------------------------
process_one_dataset <- function(ds_info) {
  id     <- ds_info$id
  in_fn  <- ds_info$input_file
  subdir <- ds_info$subdir
  
  message("======================================================")
  message("[INFO] Dataset ", id, " (", in_fn, ")")
  message("======================================================")
  
  ## 入力ファイルパス
  in_path <- file.path(input_dir, in_fn)
  if (!file.exists(in_path)) {
    stop("[ERROR] Input file not found: ", in_path)
  }
  
  ## 出力ディレクトリ作成
  out_dir <- file.path(out_root_var, subdir)
  if (!dir.exists(out_dir)) {
    dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
  }
  
  ## 1) データ読み込み
  message("[INFO] Reading data: ", in_path)
  df <- read.csv(in_path, header = TRUE, stringsAsFactors = FALSE, check.names = FALSE)
  
  ## 数値列のみ抽出（サンプル版STEP1と同様の方針）
  df_num <- select_numeric_columns(df)
  message("[INFO] n_samples = ", nrow(df_num), ", n_variables = ", ncol(df_num))
  
  if (ncol(df_num) < 2L) {
    stop("[ERROR] At least 2 numeric variables are required for variable-wise correlation/MDS.")
  }
  
  X <- as.matrix(df_num)
  var_names <- colnames(X)
  N <- nrow(X)
  
  ## 2) 変数間相関行列（variable-variable）
  message("[INFO] Computing variable-wise correlation (pairwise.complete.obs)")
  corVar <- suppressWarnings(cor(X, use = "pairwise.complete.obs"))
  rownames(corVar) <- var_names
  colnames(corVar) <- var_names
  
  ## 3) 生距離行列 d_raw_var = 1 - r (符号付きのまま)
  message("[INFO] Computing raw distance matrix d_raw_var = 1 - r")
  d_raw_var <- 1 - corVar
  rownames(d_raw_var) <- var_names
  colnames(d_raw_var) <- var_names
  
  ## 4) 欠損補正距離 d_corr_var
  message("[INFO] Computing effective sample counts for each variable pair")
  ## n_eff_var[m, n] = 変数m,nともに非欠損なサンプル数
  non_na <- !is.na(X)
  n_eff_var <- crossprod(non_na * 1L)  ## P x P
  rownames(n_eff_var) <- var_names
  colnames(n_eff_var) <- var_names
  
  message("[INFO] Applying missingness-based correction to distances")
  d_corr_var <- d_raw_var
  
  ## スケール係数 (N / n_eff_var) を計算（n_eff_var > 0 のみ）
  scale_mat <- matrix(NA_real_, nrow = ncol(X), ncol = ncol(X))
  scale_mat[n_eff_var > 0] <- N / n_eff_var[n_eff_var > 0]
  rownames(scale_mat) <- var_names
  colnames(scale_mat) <- var_names
  
  ## 補正距離
  d_corr_var <- d_corr_var * scale_mat
  
  ## 無限大やNaNは一旦NAに
  d_corr_var[!is.finite(d_corr_var)] <- NA_real_
  
  ## 対角成分は0に固定
  diag(d_corr_var) <- 0
  
  ## まだNAがあれば、非NAの最大値で埋める（その後 0〜2 にクランプ）
  upper_idx <- upper.tri(d_corr_var)
  max_val <- suppressWarnings(max(d_corr_var[upper_idx], na.rm = TRUE))
  if (!is.finite(max_val)) {
    max_val <- 2.0
  }
  na_off_diag <- is.na(d_corr_var) & !diag(ncol(d_corr_var))
  d_corr_var[na_off_diag] <- max_val
  
  ## 対称性の確保
  d_corr_var <- (d_corr_var + t(d_corr_var)) / 2
  diag(d_corr_var) <- 0
  
  ## 0〜2 にクランプ
  d_corr_var <- pmin(pmax(d_corr_var, 0), 2)
  
  rownames(d_corr_var) <- var_names
  colnames(d_corr_var) <- var_names
  
  ## 5) 保存: raw / 補正距離
  raw_path  <- file.path(out_dir, paste0("distance_raw_var_",  id, "_", date_tag, ".rds"))
  corr_path <- file.path(out_dir, paste0("distance_corr_var_", id, "_", date_tag, ".rds"))
  
  saveRDS(d_raw_var,  file = raw_path)
  saveRDS(d_corr_var, file = corr_path)
  message("[SAVED] Raw variable distance:      ", raw_path)
  message("[SAVED] Corrected variable distance:", corr_path)
  
  ## --------------------------------------------------------
  ## 6) metric MDS (cmdscale)
  ## --------------------------------------------------------
  k_mds <- min(k_mds_default, ncol(X) - 1L)
  if (k_mds < 1L) {
    stop("[ERROR] Not enough variables for MDS (k_mds < 1).")
  }
  
  message("[INFO] Running cmdscale (metric MDS) with k = ", k_mds)
  
  dist_obj <- as.dist(d_corr_var)
  
  cmd_res <- cmdscale(dist_obj, k = k_mds, eig = TRUE)
  coords_cmd <- cmd_res$points
  eig_vals   <- cmd_res$eig
  
  rownames(coords_cmd) <- var_names
  colnames(coords_cmd) <- paste0("Dim", seq_len(ncol(coords_cmd)))
  
  ## 固有値CSV
  eig_df <- data.frame(
    component = seq_along(eig_vals),
    eigenvalue = eig_vals,
    stringsAsFactors = FALSE
  )
  
  eig_csv_path <- file.path(out_dir, paste0("eigvals_cmdscale_var_", id, "_", date_tag, ".csv"))
  write.csv(eig_df, eig_csv_path, row.names = FALSE)
  message("[SAVED] cmdscale eigenvalues (CSV): ", eig_csv_path)
  
  ## cmdscale座標RDS
  cmd_rds_path <- file.path(out_dir, paste0("mds_cmdscale_var_", id, "_", date_tag, ".rds"))
  saveRDS(coords_cmd, cmd_rds_path)
  message("[SAVED] cmdscale coordinates (RDS): ", cmd_rds_path)
  
  ## --------------------------------------------------------
  ## 7) non-metric MDS (isoMDS)
  ## --------------------------------------------------------
  message("[INFO] Running isoMDS (non-metric MDS) with k = ", k_mds,
          " using cmdscale coordinates as initial configuration")
  
  iso_out <- NULL
  iso_ok <- TRUE
  iso_msg <- NULL
  
  iso_out <- tryCatch(
    {
      MASS::isoMDS(d = dist_obj, y = coords_cmd, k = k_mds, maxit = 200, trace = FALSE)
    },
    error = function(e) {
      iso_ok <<- FALSE
      iso_msg <<- conditionMessage(e)
      NULL
    },
    warning = function(w) {
      ## warningはメッセージだけ出して続行
      message("[WARN] isoMDS warning: ", conditionMessage(w))
      invokeRestart("muffleWarning")
    }
  )
  
  if (!iso_ok || is.null(iso_out)) {
    message("[ERROR] isoMDS failed for dataset ", id, ": ", iso_msg)
  } else {
    coords_iso <- iso_out$points
    rownames(coords_iso) <- var_names
    colnames(coords_iso) <- paste0("Dim", seq_len(ncol(coords_iso)))
    
    stress_val <- iso_out$stress
    
    iso_rds_path <- file.path(out_dir, paste0("mds_isoMDS_var_", id, "_", date_tag, ".rds"))
    saveRDS(coords_iso, iso_rds_path)
    message("[SAVED] isoMDS coordinates (RDS): ", iso_rds_path)
    
    stress_df <- data.frame(stress = stress_val)
    stress_csv_path <- file.path(out_dir, paste0("stress_isoMDS_var_", id, "_", date_tag, ".csv"))
    write.csv(stress_df, stress_csv_path, row.names = FALSE)
    message("[SAVED] isoMDS stress (CSV):      ", stress_csv_path)
  }
  
  message("[INFO] Finished dataset ", id)
}

## ------------------------------------------------------------
## 全データセット(A〜D)をループ処理
## ------------------------------------------------------------
for (nm in names(datasets)) {
  ds <- datasets[[nm]]
  process_one_dataset(ds)
}

message("======================================================")
message("[INFO] All datasets (A-D) finished.")
message("======================================================")



[INFO] Dataset A (preprocessed_features_OH.csv)


[INFO] Reading data: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/for_MDS_STEP2/preprocessed_features_OH.csv

[INFO] n_samples = 1046, n_variables = 395

[INFO] Computing variable-wise correlation (pairwise.complete.obs)

[INFO] Computing raw distance matrix d_raw_var = 1 - r

[INFO] Computing effective sample counts for each variable pair

[INFO] Applying missingness-based correction to distances

[SAVED] Raw variable distance:      /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core_var/01_distance_mds_variables/A_OH_plus_others/distance_raw_var_A_20251128.rds

[SAVED] Corrected variable distance:/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core_var/01_distance_mds_vari

[INFO] Running cmdscale (metric MDS) with k = 10

[SAVED] cmdscale eigenvalues (CSV): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core_var/01_distance_mds_variables/D_FP_only/eigvals_cmdscale_var_D_20251128.csv

[SAVED] cmdscale coordinates (RDS): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core_var/01_distance_mds_variables/D_FP_only/mds_cmdscale_var_D_20251128.rds

[INFO] Running isoMDS (non-metric MDS) with k = 10 using cmdscale coordinates as initial configuration

[SAVED] isoMDS coordinates (RDS): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core_var/01_distance_mds_variables/D_FP_only/mds_isoMDS_var_D_20251128.rds

[SAVED] isoMDS stress (CSV):      /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No